# Import Libraries

Here is a breakdown of libraries we will be using to perform preprocessing, data analysis, training the model, etc.

In [1]:
# communicating with the os (specifically for file and directory management)
import os

# math and data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# extracting audio features
import librosa

# data cleaning and splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# saving models
import pickle

# Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets/MusicMaster"
data = []

# just getting all the files from the drive
for genre in os.listdir(DATASET_PATH):
    genre_path = os.path.join(DATASET_PATH, genre)
    if not os.path.isdir(genre_path):
        continue

    for file in os.listdir(genre_path):
        if file.endswith(".mp3"):
            file_path = os.path.join(genre_path, file)
            data.append({
                "filepath": file_path,
                "label": genre
            })

# store filepaths with corresponding label/genre (will extract actual file later)
df = pd.DataFrame(data)
df.tail()

In [ ]:
def load_audio(file_path, sr=48000, duration=None, offset=0.0):
    try:
        signal, _ = librosa.load(
            file_path,
            sr=sr,
            mono=True,
            offset=offset,
            duration=duration
        )
        return signal
    except Exception as e:
        print(f"Failed to load: {file_path}: {e}")
        return None

In [ ]:
signal = load_audio(df.iloc[0]["filepath"])
print(signal.shape)

In [ ]:
SR = 48000
CHUNK_DURATION = 3
OVERLAP = 0.5

In [ ]:
def split_signal(signal, sr=SR, chunk_duration=CHUNK_DURATION, overlap=OVERLAP):
    chunk_length = int(sr * chunk_duration)
    step = int(chunk_length * (1 - overlap))
    chunks = []

    for start in range(0, len(signal) - chunk_length, step):
        chunk = signal[start:start + chunk_length]
        chunks.append(chunk)

    return chunks

In [8]:
N_MELS = 40
N_FFT = 2048
HOP_LENGTH = 512
MAX_CHUNKS_PER_FILE = 20

In [ ]:
def extract_mel_spectrogram(signal, sr):
    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

In [ ]:
X = []
y = []

import matplotlib.pyplot as plt
import librosa.display

for idx, row in df.iterrows():

    file_path = row["filepath"]
    label = row["label"]

    offset = 0.0
    chunks_loaded = 0

    while chunks_loaded < MAX_CHUNKS_PER_FILE:
        signal = load_audio(
            file_path,
            sr=SR,
            duration=CHUNK_DURATION,
            offset=offset
        )

        if signal is None or len(signal) < SR * CHUNK_DURATION:
            break  # end of file or bad read

        mel_spec = extract_mel_spectrogram(signal, SR)


        X.append(mel_spec)
        y.append(label)

        offset += CHUNK_DURATION
        chunks_loaded += 1

    if idx % 10 == 0:
        print(f"Processed {idx}/{len(df)} files")

plt.figure(figsize=(10, 4))
librosa.display.specshow(
    X[16],
    sr=SR,
    hop_length=512,
    x_axis='time',
    y_axis='mel'
)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_onehot = to_categorical(y_encoded)

num_classes = y_onehot.shape[1]
print("Classes:", label_encoder.classes_)

In [ ]:
X = X[..., np.newaxis]
print("Final X shape:", X.shape)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [15]:
def modelInit(shape = (224, 224, 3), num_classes = 16):
  inputs = layers.Input(shape=shape)
  x = layers.Conv2D(64, 7, strides=2, padding='same')(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.ReLU()(x)
  x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

  #ResNet-34 layout: (3, 4, 6, 3) blocks
  block_layers = [64, 128, 192, 256]
  blocks_per_stage = [3, 4, 6, 3]

  for filters, n_blocks in zip(block_layers, blocks_per_stage):
    for i in range(n_blocks):
      stride = 2 if i == 0 and filters != 64 else 1
      x = residualBlock(x, filters, stride=stride)

  x = layers.GlobalAveragePooling2D()(x)
  outputs = layers.Dense(num_classes, activation="softmax")(x)

  model = Model(inputs, outputs)
  return model

def residualBlock(x, filters, stride=1):
  skip = x

  x = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
  x = layers.BatchNormalization()(x)
  x = layers.ReLU()(x)

  x = layers.Conv2D(filters, 3, strides=1, padding='same')(x)
  x = layers.BatchNormalization()(x)

  #Match shapes if stride changes or channel count changes
  if skip.shape[-1] != filters or stride != 1:
    skip = layers.Conv2D(filters, 1, strides=stride, padding='same')(skip)
    skip = layers.BatchNormalization()(skip)

  #Add both channels
  x = layers.Add()([x, skip])
  x = layers.ReLU()(x)
  return x

In [ ]:
model = modelInit(shape=X_train.shape[1:], num_classes=10)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

# Save The Models

In [ ]:
MODEL_PATH = "/content/genre_cnn_model.keras"
ENCODER_PATH = "/content/genre_label_encoder.pkl"

# Save model
model.save(MODEL_PATH)

# Save encoder
with open(ENCODER_PATH, "wb") as f:
    pickle.dump(label_encoder, f)

print("Saved locally in /content/")

In [29]:
from google.colab import files

files.download(MODEL_PATH)
files.download(ENCODER_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>